In [0]:
%sql
-- Table 1: customer
CREATE TABLE IF NOT EXISTS oms.test.customer (
  customer_id INT,
  customer_name STRING,
  city STRING,
  age INT,
  country STRING
);

INSERT INTO oms.test.customer VALUES (1, 'Alice', 'Pune', 30, 'India');

-- Table 2: contact
CREATE TABLE IF NOT EXISTS oms.test.contact (
  customer_id INT,
  email STRING,
  phone STRING,
  dob DATE,
  gender STRING
);

INSERT INTO oms.test.contact VALUES (1, 'alice@example.com', '1234567890', '1993-06-01', 'F');

-- Table 3: employee
CREATE TABLE IF NOT EXISTS oms.test.employee (
  employee_id INT,
  department STRING,
  salary INT,
  manager STRING,
  joining_date DATE
);

INSERT INTO oms.test.employee VALUES (1, 'IT', 60000, 'Bob', '2020-01-10');

-- Table 4: region_info
CREATE TABLE IF NOT EXISTS oms.test.region_info (
  customer_id INT,
  region STRING,
  zone STRING,
  score INT,
  status STRING
);

INSERT INTO oms.test.region_info VALUES (1, 'West', 'Zone1', 90, 'Active');


In [0]:
%sql
CREATE OR REPLACE VIEW oms.test.vw_customer_overview AS
SELECT
  c.customer_id,
  c.customer_name,
  ct.email,
  e.department,
  r.region
FROM oms.test.customer c
JOIN oms.test.contact ct ON c.customer_id = ct.customer_id
JOIN oms.test.employee e ON c.customer_id = e.employee_id
JOIN oms.test.region_info r ON c.customer_id = r.customer_id;


select * from oms.test.vw_customer_overview ;

In [0]:

view_df = spark.table("oms.test.vw_customer_overview")

# Load base tables
customer_df = spark.table("oms.test.customer")
contact_df = spark.table("oms.test.contact")
employee_df = spark.table("oms.test.employee")
region_df = spark.table("oms.test.region_info")

#  join
manual_join_df = (
    customer_df.alias("c")
    .join(contact_df.alias("ct"), "customer_id")
    .join(employee_df.alias("e"), customer_df["customer_id"] == employee_df["employee_id"])
    .join(region_df.alias("r"), "customer_id")
    .select(
        customer_df["customer_id"],
        customer_df["customer_name"],
        contact_df["email"],
        employee_df["department"],
        region_df["region"]
    )
)

# Compare counts
view_count = view_df.count()
manual_count = manual_join_df.count()

print(f"View record count     : {view_count}")
print(f"Manual join count     : {manual_count}")
print(" Count Match" if view_count == manual_count else " Count Mismatch")

# Check data differences
diff_df = view_df.exceptAll(manual_join_df)

# Show differences if any
if diff_df.count() == 0:
    print("Data match: No differences found between view and manual join.")
else:
    print(" Data mismatch detected. Differences:")
    diff_df.show(truncate=False)


In [0]:
view_columns = view_df.columns
manual_columns = manual_join_df.columns

# Ensure columns match in name and order
if view_columns != manual_columns:
    print("Column mismatch found between view and manual join.")
    print("View columns     :", view_columns)
    print("Manual columns   :", manual_columns)
else:
    for col in view_columns:
        diff = view_df.select(col).subtract(manual_join_df.select(col)).union(
               manual_join_df.select(col).subtract(view_df.select(col)))
        
        if diff.count() > 0:
            print(f"Data mismatch in column: {col}")
            diff.show(truncate=False)
        else:
            print(f"Column {col}: Match ✅")


In [0]:
diff.show()